In [ ]:
from pyspark.sql import functions as Fschema = "workspace.retail_lakehouse"# every gold table aggregates off the enriched silver fact tableorders_enriched = spark.table(f"{schema}.silver_orders_enriched")

In [ ]:
# --- revenue by month ---gold_revenue_by_month = (    orders_enriched    .withColumn("year_month", F.date_format("order_date", "yyyy-MM"))    .groupBy("year_month")    .agg(        F.sum("amount").alias("total_revenue"),        F.count("order_id").alias("num_orders")    )    .orderBy("year_month"))(gold_revenue_by_month.write.format("delta").mode("overwrite")    .saveAsTable(f"{schema}.gold_revenue_by_month"))gold_revenue_by_month.show()

In [ ]:
# --- revenue by state ---gold_revenue_by_state = (    orders_enriched    .groupBy("state")    .agg(F.sum("amount").alias("total_revenue"), F.count("order_id").alias("num_orders"))    .orderBy(F.desc("total_revenue")))(gold_revenue_by_state.write.format("delta").mode("overwrite")    .saveAsTable(f"{schema}.gold_revenue_by_state"))

In [ ]:
# --- revenue by category ---gold_revenue_by_category = (    orders_enriched    .groupBy("category")    .agg(F.sum("amount").alias("total_revenue"), F.count("order_id").alias("num_orders"))    .orderBy(F.desc("total_revenue")))(gold_revenue_by_category.write.format("delta").mode("overwrite")    .saveAsTable(f"{schema}.gold_revenue_by_category"))

In [ ]:
# --- top 10 products by revenue ---gold_top_products = (    orders_enriched    .groupBy("product_id", "category", "brand")    .agg(F.sum("amount").alias("total_revenue"), F.count("order_id").alias("num_orders"))    .orderBy(F.desc("total_revenue"))    .limit(10))(gold_top_products.write.format("delta").mode("overwrite")    .saveAsTable(f"{schema}.gold_top_products"))

In [ ]:
# --- average basket value ---gold_avg_basket_value = orders_enriched.agg(    F.avg("amount").alias("avg_order_value"),    F.count("order_id").alias("total_orders"))(gold_avg_basket_value.write.format("delta").mode("overwrite")    .saveAsTable(f"{schema}.gold_avg_basket_value"))

In [ ]:
# --- top 10 customers by spend ---gold_top_customers = (    orders_enriched    .groupBy("customer_id", "city", "state")    .agg(F.sum("amount").alias("total_spent"), F.count("order_id").alias("num_orders"))    .orderBy(F.desc("total_spent"))    .limit(10))(gold_top_customers.write.format("delta").mode("overwrite")    .saveAsTable(f"{schema}.gold_top_customers"))

In [ ]:
# --- repeat customers (more than one order) ---gold_repeat_customers = (    orders_enriched    .groupBy("customer_id")    .agg(        F.countDistinct("order_id").alias("num_orders"),        F.sum("amount").alias("total_spent")    )    .filter(F.col("num_orders") > 1)    .orderBy(F.desc("num_orders")))(gold_repeat_customers.write.format("delta").mode("overwrite")    .saveAsTable(f"{schema}.gold_repeat_customers"))

In [ ]:
# --- verify ---for t in ["gold_revenue_by_month", "gold_revenue_by_state", "gold_revenue_by_category",          "gold_top_products", "gold_top_customers", "gold_avg_basket_value", "gold_repeat_customers"]:    n = spark.table(f"{schema}.{t}").count()    print(f"{t}: {n} rows")